# 🌌 Cosmic Detector
### One deep-learning model that finds **exoplanets**, **black-hole candidates** and **asteroids** in real telescope light curves

---
**The idea in one sentence:** a *light curve* is a record of how bright something looks over time. Planets, black holes and asteroids each leave a different fingerprint in a light curve, so **one** neural network can learn to recognise all three.

| What happens in space | Fingerprint in the light curve | Real data source (merged) |
|---|---|---|
| 🪐 A planet passes in front of its star | Small, short, **repeating dips** | NASA **Kepler** telescope (Kaggle) |
| 🕳️ A dark object (often a star, sometimes a **black hole**) passes in front of a distant star and bends its light (gravitational microlensing) | One smooth, symmetric **brightening** lasting days to months. **Very long** events point to heavy, dark lenses: black-hole candidates | **OGLE-IV** survey, Warsaw University (Mróz et al. 2019) |
| ☄️ An asteroid spins | A **wave**, usually two bumps per rotation, every few hours | **ALCDEF** database, NASA Planetary Data System |
| ⭕ Ordinary stars, variable stars, binaries | Anything else | Kepler + OGLE non-event stars |

**Honesty rules used throughout this notebook**
1. Only **real observed data**. Nothing is simulated.
2. Data is split **by object**: one star or asteroid is never in both training and testing.
3. The Kepler test set is the official, untouched one.
4. We check whether the model is cheating by recognising the *telescope* instead of the *physics*, and we report the weak spots.

### ▶️ How to run
**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. It takes about 15–25 minutes.

In [ ]:
# ⚙️ SETTINGS
EPOCHS        = 30
BATCH         = 128
MAX_ASTEROID_SESSIONS = 7000      # asteroid light curves to use (more = slower)
MAX_OGLE_EVENTS       = 6000      # OGLE stars to use
PLANET_AUG_COPIES     = 40        # augmented copies per training planet (only 37 real ones exist!)
RUN_ABLATION  = True              # also train a model WITHOUT the duration input (honesty check)
SEED          = 42

## 0. Setup

In [ ]:
!pip -q install astropy gradio kaggle

In [ ]:
import os, sys, json, shutil, subprocess, tarfile, zipfile, urllib.request, warnings, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (classification_report, confusion_matrix, f1_score, balanced_accuracy_score,
                             average_precision_score, precision_recall_fscore_support)
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
COLORS = {"Other": "#8b93a7", "Planet": "#2dd4bf", "Lensing": "#a78bfa", "Asteroid": "#f59e0b"}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "none (slower, still works)")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR, DATA_DIR, IN_COLAB = "/content/drive/MyDrive/CosmicML", "/content/drive/MyDrive/CosmicData", True
except Exception:
    SAVE_DIR, DATA_DIR, IN_COLAB = os.environ.get("COSMIC_SAVE", "./CosmicML"), os.environ.get("COSMIC_DATA", "./CosmicData"), False
os.makedirs(SAVE_DIR, exist_ok=True); os.makedirs(DATA_DIR, exist_ok=True)
print("Data folder:", DATA_DIR, "| outputs:", SAVE_DIR)

## 1. Get the three real datasets
Everything downloads automatically from the original sources and is cached in your Google Drive (`MyDrive/CosmicData`), so the next run is instant.

| Dataset | Source | Size |
|---|---|---|
| Kepler light curves (`exoTrain.csv`, `exoTest.csv`) | Kaggle: `keplersky/kepler-labelled-time-series-data` (needs your free `kaggle.json`) | 260 MB |
| OGLE-IV microlensing events (`phot.tar.gz`, `table3.dat`) | ftp.astrouw.edu.pl/ogle/ogle4/microlensing_maps | 50 MB |
| ALCDEF asteroid light curves | sbnarchive.psi.edu (NASA PDS) | 104 MB zip |

💡 If a download fails, put the files in `MyDrive/CosmicData` yourself and re-run. The notebook looks there first.

In [ ]:
def download(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return dest
    print("⬇️ ", url)
    urllib.request.urlretrieve(url, dest + ".part"); os.replace(dest + ".part", dest)
    return dest

# --- Kepler (Kaggle) ---
KEP_TRAIN, KEP_TEST = os.path.join(DATA_DIR, "exoTrain.csv"), os.path.join(DATA_DIR, "exoTest.csv")
if not os.path.exists(KEP_TRAIN):
    kj = os.path.expanduser("~/.kaggle/kaggle.json"); os.makedirs(os.path.dirname(kj), exist_ok=True)
    if os.path.exists(os.path.join(DATA_DIR, "kaggle.json")):
        shutil.copy(os.path.join(DATA_DIR, "kaggle.json"), kj)
    elif IN_COLAB:
        from google.colab import files
        print("👉 Upload kaggle.json (Kaggle → Settings → API → Create New Token)")
        open(kj, "wb").write(list(files.upload().values())[0]); shutil.copy(kj, os.path.join(DATA_DIR, "kaggle.json"))
    os.chmod(kj, 0o600)
    subprocess.run(["kaggle", "datasets", "download", "-d", "keplersky/kepler-labelled-time-series-data",
                    "-p", DATA_DIR, "--unzip"], check=True)
print("Kepler ✅", os.path.exists(KEP_TRAIN) and os.path.exists(KEP_TEST))

# --- OGLE-IV microlensing ---
OGLE_BASE = "https://ftp.astrouw.edu.pl/ogle/ogle4/microlensing_maps/"
for alt in ["table3.dat.txt", "table3.txt"]:                 # accept the file however your browser saved it
    if os.path.exists(os.path.join(DATA_DIR, alt)) and not os.path.exists(os.path.join(DATA_DIR, "table3.dat")):
        shutil.copy(os.path.join(DATA_DIR, alt), os.path.join(DATA_DIR, "table3.dat"))
TABLE3 = download(OGLE_BASE + "table3.dat", os.path.join(DATA_DIR, "table3.dat"))
PHOT_DIR = os.path.join(DATA_DIR, "ogle_phot")
if not os.path.exists(PHOT_DIR):
    local = [os.path.join(DATA_DIR, f) for f in ("phot.tar.gz", "phot.tar") if os.path.exists(os.path.join(DATA_DIR, f))]
    PHOT_TGZ = local[0] if local else download(OGLE_BASE + "phot.tar.gz", os.path.join(DATA_DIR, "phot.tar.gz"))
    with tarfile.open(PHOT_TGZ) as tf_:                     # works for .tar and .tar.gz
        tf_.extractall(PHOT_DIR)
print("OGLE ✅", sum(len(f) for _, _, f in os.walk(PHOT_DIR)), "light-curve files")

# --- ALCDEF asteroids (NASA PDS) ---
ALC_DIR = os.path.join(DATA_DIR, "alcdef")
if os.path.exists(os.path.join(DATA_DIR, "gbo.ast.alcdef-database_V1_0")):   # folder uploaded by hand
    ALC_DIR = os.path.join(DATA_DIR, "gbo.ast.alcdef-database_V1_0")
if not os.path.exists(ALC_DIR):
    z = download("https://sbnarchive.psi.edu/pds4/non_mission/gbo.ast.alcdef-database_V1_0.zip",
                 os.path.join(DATA_DIR, "alcdef.zip"))
    with zipfile.ZipFile(z) as zf:
        zf.extractall(ALC_DIR)
print("ALCDEF ✅", sum(1 for r, _, f in os.walk(ALC_DIR) for x in f if x.startswith("alcdef_lcdata")), "data files")

## 2. The shared toolkit
**The key design rule: every light curve, from every telescope, goes through exactly the same steps.** Otherwise the model could learn "this was processed differently" instead of the physics.
1. Convert to brightness (up = brighter). Magnitudes are converted to relative flux.
2. Remove single-point spikes (cosmic rays, glitches).
3. Average into **1,024 equal time bins** and fill gaps with matching noise.
4. **Channel A** = the whole curve (for slow events: lensing, rotation). **Channel B** = a high-pass filtered copy (for fast events: transit dips).
5. Divide by the noise level, then compress big values with sign·log(1+|x|).

The model also receives one number: **log10(duration in days)**. Timescale is real physics: transits last hours, lensing lasts weeks, and asteroids rotate in hours. Later we test how much the model relies on it.

*(These cells are saved as `.py` files in Drive so the app at the end uses the identical code.)*

In [ ]:
%%writefile $SAVE_DIR/cosmic_core.py
"""Cosmic Detector core: ONE preprocessing pipeline for every light curve, whatever telescope it came from.

Classes:
    0 = Other       (ordinary / variable stars, eclipsing binaries, telescope noise)
    1 = Planet      (exoplanet transit)
    2 = Lensing     (gravitational microlensing; long events are black-hole candidates)
    3 = Asteroid    (asteroid rotation light curve)
"""
import numpy as np

CLASSES = ["Other", "Planet transit", "Microlensing (black-hole candidate if long)", "Asteroid"]
SHORT = ["Other", "Planet", "Lensing", "Asteroid"]
L = 1024                      # every light curve is resampled to this many points


def _running_median(x, w=5):
    from scipy.ndimage import median_filter
    return median_filter(x, size=w, mode="nearest")


def to_brightness(values, kind):
    """Return a 'brightness' series where UP = brighter, for any input type.
    kind = 'flux' (Kepler) or 'mag' (OGLE, ALCDEF; smaller magnitude = brighter)."""
    v = np.asarray(values, dtype=np.float64)
    if kind == "mag":
        return 10 ** (-0.4 * (v - np.nanmedian(v))) - 1.0      # relative flux change
    return v - np.nanmedian(v)


def bin_to_grid(t, y, n=L, seed=0):
    """Average irregular samples into n equal time bins; fill empty bins by interpolation."""
    t = np.asarray(t, float); y = np.asarray(y, float)
    ok = np.isfinite(t) & np.isfinite(y)
    t, y = t[ok], y[ok]
    if len(t) < 5:
        return None
    edges = np.linspace(t.min(), t.max() + 1e-9, n + 1)
    idx = np.clip(np.searchsorted(edges, t, side="right") - 1, 0, n - 1)
    s = np.bincount(idx, weights=y, minlength=n); c = np.bincount(idx, minlength=n)
    g = np.full(n, np.nan); g[c > 0] = s[c > 0] / c[c > 0]
    good = np.isfinite(g)
    centers = np.arange(n)
    g[~good] = np.interp(centers[~good], centers[good], g[good])
    if (~good).any():
        # Fill gaps with interpolation PLUS noise at the data's own noise level, so that gaps
        # (e.g. OGLE's seasonal breaks) don't look like suspiciously perfect flat lines.
        gg = g[good]
        noise = 1.4826 * np.median(np.abs(np.diff(gg) - np.median(np.diff(gg)))) / np.sqrt(2) if len(gg) > 3 else 0.0
        rng = np.random.default_rng(seed)
        g[~good] += rng.normal(0, noise, (~good).sum())
    return g, good.astype(np.float32)


def preprocess(t, values, kind, seed=0):
    """Raw (time [days], value) → model input (L, 2) + duration feature.

    Identical for every source:
      1. convert to brightness (up = brighter)
      2. remove single-point spikes (5-point running median, > 5 sigma)
      3. bin onto a fixed grid of L points (gaps filled with interpolation + matching noise)
      4. channel A = whole curve  (slow events: lensing, rotation)
         channel B = high-pass residual, window = 5% of the curve (fast events: transits)
      5. divide by the point-to-point noise level, then compress with sign·log(1+|x|)
    """
    from scipy.signal import savgol_filter
    t = np.asarray(t, float); y = to_brightness(values, kind)
    o = np.argsort(t); t, y = t[o], y[o]
    ok = np.isfinite(t) & np.isfinite(y); t, y = t[ok], y[ok]
    if len(t) < 10:
        return None
    rm = _running_median(y, 5)
    resid = y - rm
    s = 1.4826 * np.median(np.abs(resid - np.median(resid))) + 1e-12
    y = np.where(np.abs(resid) > 5 * s, rm, y)
    out = bin_to_grid(t, y, L, seed)
    if out is None:
        return None
    g, cov = out
    w = max(5, int(0.05 * L) | 1)
    hp = g - savgol_filter(g, w, 2)
    noise = 1.4826 * np.median(np.abs(np.diff(g) - np.median(np.diff(g)))) / np.sqrt(2) + 1e-12
    a = (g - np.median(g)) / noise
    b = hp / noise
    comp = lambda z: np.sign(z) * np.log1p(np.abs(z))
    X = np.stack([comp(a), comp(b)], axis=-1).astype(np.float32)
    duration = float(t.max() - t.min())
    return X, np.float32(np.log10(max(duration, 1e-3)))


# ---------- physics stage: extra measurements once the class is known ----------

def paczynski(t, t0, tE, u0, fs, base):
    """Point-lens microlensing magnification curve (Paczyński 1986), in flux units."""
    u = np.sqrt(u0 ** 2 + ((t - t0) / tE) ** 2)
    A = (u ** 2 + 2) / (u * np.sqrt(u ** 2 + 4))
    return base * (fs * A + (1 - fs))


def fit_lens(t, mag):
    """Fit the Paczyński model to a lensing light curve. Returns dict with tE (days)."""
    from scipy.optimize import curve_fit
    t = np.asarray(t, float); mag = np.asarray(mag, float)
    f = 10 ** (-0.4 * (mag - np.median(mag)))
    base = np.median(f)
    t0 = t[np.argmax(_running_median(f, 9))]
    best = None
    for tE0 in [5, 20, 60, 150]:
        for u0 in [0.05, 0.3, 0.8]:
            try:
                p, _ = curve_fit(paczynski, t, f, p0=[t0, tE0, u0, 1.0, base],
                                 bounds=([t.min(), 0.5, 1e-4, 0.01, 0], [t.max(), 1000, 3, 1.0, np.inf]), maxfev=4000)
                r = np.sum((f - paczynski(t, *p)) ** 2)
                if best is None or r < best[1]:
                    best = (p, r)
            except Exception:
                pass
    if best is None:
        return None
    t0, tE, u0, fs, base = best[0]
    return {"t0": float(t0), "tE_days": float(tE), "u0": float(u0), "fs": float(fs), "params": best[0]}


def rotation_period_hours(t, mag):
    """Asteroid rotation period from a Lomb-Scargle periodogram (asteroids show 2 peaks per rotation)."""
    from astropy.timeseries import LombScargle
    t = np.asarray(t, float); span_h = (t.max() - t.min()) * 24
    if span_h < 1:
        return None
    dt = np.median(np.diff(np.sort(t)))
    f_max = min(48.0, 0.5 / max(dt, 1e-6))                     # stay below the Nyquist limit → no aliasing
    freq = np.linspace(1 / (span_h / 24), f_max, 4000)         # cycles per day
    p = LombScargle(t, mag).power(freq)
    f_best = freq[np.argmax(p)]
    return float(2 * 24 / f_best)                             # double-peaked → rotation = 2 × light-curve period


def transit_period_days(t, flux):
    """Orbital period from Box Least Squares (only meaningful for planet-class curves)."""
    from astropy.timeseries import BoxLeastSquares
    t = np.asarray(t, float); y = np.asarray(flux, float)
    from scipy.signal import savgol_filter
    y = y - savgol_filter(y, 101 if len(y) > 101 else (len(y) // 2) * 2 - 1, 2)
    span = t.max() - t.min()
    periods = np.exp(np.linspace(np.log(0.5), np.log(span / 2.2), 4000))
    r = BoxLeastSquares(t, y).power(periods, [0.05, 0.08, 0.12, 0.2], objective="snr")
    k = np.argmax(r.power)
    return float(r.period[k]), float(r.depth[k]), float(r.power[k])


# ---------- readers for the four real data formats ----------

def read_ogle_phot(path_or_text):
    """OGLE photometry: columns HJD, I_mag, err (extra columns ignored)."""
    import io
    src = io.StringIO(path_or_text) if "\n" in str(path_or_text) else path_or_text
    d = np.loadtxt(src, usecols=(0, 1, 2), ndmin=2)
    d = d[(d[:, 1] > 0) & (d[:, 1] < 30)]
    return d[:, 0], d[:, 1]


def read_two_columns(path):
    """Generic file: time (days, e.g. JD) and magnitude or flux in the first two numeric columns."""
    import pandas as pd
    df = pd.read_csv(path, sep=None, engine="python", comment="#", header=None)
    df = df.apply(lambda c: pd.to_numeric(c, errors="coerce")).dropna(axis=1, how="all").dropna()
    return df.iloc[:, 0].values, df.iloc[:, 1].values

In [ ]:
%%writefile $SAVE_DIR/cosmic_data.py
"""Loaders that turn the three REAL datasets into one merged list of light curves.
Each item: dict(t=days, v=values, kind='flux'|'mag', label=int, group=str, source=str, split='train'|'val'|'test')"""
import glob, io, os, zlib
import numpy as np
import pandas as pd

KEPLER_CADENCE_DAYS = 29.4244 / 60 / 24


def split_of(group, val=0.15, test=0.15):
    """Deterministic split by object: every light curve of one star/asteroid lands in the same split."""
    h = (zlib.crc32(str(group).encode()) % 10000) / 10000
    return "test" if h < test else ("val" if h < test + val else "train")


# ---------------- 1. Kepler (Kaggle "Exoplanet Hunting in Deep Space") ----------------
def load_kepler(train_csv, test_csv, seed=0):
    items = []
    rng = np.random.default_rng(seed)
    for path, part in [(train_csv, "trainval"), (test_csv, "test")]:
        df = pd.read_csv(path)
        y = (df["LABEL"].values == 2).astype(int)
        X = df.drop(columns=["LABEL"]).values.astype(np.float64)
        t = np.arange(X.shape[1]) * KEPLER_CADENCE_DAYS
        for i in range(len(df)):
            if part == "test":
                split = "test"
            else:   # stratified 85/15 train/val split, planets spread evenly
                split = "val" if rng.random() < 0.15 else "train"
            items.append(dict(t=t, v=X[i], kind="flux", label=1 if y[i] else 0,
                              group=f"kepler_{part}_{i}", source="Kepler", split=split))
    return items


# ---------------- 2. ALCDEF asteroid light curves (NASA PDS) ----------------
def _skip_to_data(path):
    with open(path, encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            if line.startswith("# Data"):
                return i + 1
    return 0


def load_alcdef_metadata(path):
    with open(path, encoding="utf-8", errors="replace") as f:
        lines = f.readlines()
    hdr = [i for i, l in enumerate(lines) if l.startswith("# Column definitions")][0] + 1
    skip = [i for i, l in enumerate(lines) if l.startswith("# Data")][0] + 1
    cols = pd.read_csv(io.StringIO(lines[hdr]), header=None).iloc[0].tolist()
    md = pd.read_csv(path, skiprows=skip, header=None, names=cols, dtype=str, on_bad_lines="skip")
    return md[["ID", "Facility", "Instrument"]].rename(columns={"ID": "MDID"})


def load_alcdef(data_dir, max_sessions=7000, min_points=40, min_hours=2.0, max_days=3.0, max_err=0.05, seed=0):
    """One 'session' (a night or a TESS stretch of one asteroid) = one light curve."""
    files = sorted(glob.glob(os.path.join(data_dir, "**", "alcdef_lcdata-*.csv"), recursive=True))
    rng = np.random.default_rng(seed)
    per_file = int(np.ceil(max_sessions / max(len(files), 1) * 1.6))
    items = []
    for f in files:
        lc = pd.read_csv(f, skiprows=_skip_to_data(f), header=None,
                         names=["ObjNumber", "ObjName", "MDID", "JD", "Mag", "MagErr"],
                         dtype={"ObjName": str}, on_bad_lines="skip")
        lc = lc[(lc.Mag > -5) & (lc.Mag < 30) & np.isfinite(lc.JD)]
        try:
            md = load_alcdef_metadata(f.replace("lcdata", "metadata"))
            md["MDID"] = pd.to_numeric(md["MDID"], errors="coerce")
            fac = dict(zip(md.MDID, md.Facility.fillna("-")))
        except Exception:
            fac = {}
        g = lc.groupby("MDID")
        stats = g.agg(n=("JD", "size"), t0=("JD", "min"), t1=("JD", "max"), err=("MagErr", "median"),
                      obj=("ObjNumber", "first"), name=("ObjName", "first"))
        stats["hours"] = (stats.t1 - stats.t0) * 24
        ok = stats[(stats.n >= min_points) & (stats.hours >= min_hours) & (stats.hours <= max_days * 24)
                   & ((stats.err < max_err) | (stats.err <= 0) | stats.err.isna())]
        chosen = ok.sample(min(per_file, len(ok)), random_state=int(rng.integers(1e9))) if len(ok) else ok
        for mdid, row in chosen.iterrows():
            d = g.get_group(mdid)
            grp = f"ast_{row.obj}" if row.obj and row.obj > 0 else f"ast_{row['name']}"
            items.append(dict(t=d.JD.values - d.JD.values.min(), v=d.Mag.values, kind="mag", label=3,
                              group=grp, source="ALCDEF-" + ("TESS" if str(fac.get(mdid, "")).strip() == "TESS" else "ground"),
                              split=split_of(grp), name=f"({row.obj}) {row['name']}", mdid=int(mdid)))
    if len(items) > max_sessions:
        idx = rng.choice(len(items), max_sessions, replace=False)
        items = [items[i] for i in sorted(idx)]
    return items


# ---------------- 3. OGLE-IV microlensing (Mróz et al. 2019) ----------------
def read_table3(path):
    """Parse table3.dat using the byte positions listed in its own header (fixed-width format)."""
    import re
    specs, names = [], []
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            m = re.match(r"#\s+(\d+)-\s*(\d+)\s+\S+\s+(\S+)", line)
            if m:
                specs.append((int(m.group(1)) - 1, int(m.group(2)))); names.append(m.group(3))
    df = pd.read_fwf(path, colspecs=specs, names=names, comment="#", header=None)
    df = df[df["name"].astype(str).str.contains(r"\.")]
    for c in names:
        if c not in ("name", "field", "ews_id"):
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df["name"] = df["name"].astype(str).str.strip()
    return df.reset_index(drop=True)


def seasons(t, gap_days=60):
    """Split an OGLE light curve into observing seasons separated by gaps > gap_days."""
    cuts = np.where(np.diff(t) > gap_days)[0]
    bounds = np.concatenate([[0], cuts + 1, [len(t)]])
    return [(bounds[i], bounds[i + 1]) for i in range(len(bounds) - 1)]


def load_ogle(phot_dir, table3_path, max_events=6000, seed=0, min_points=60):
    """Lensing samples = the observing season containing the event (event at a random position).
    'Other' samples = a different season of the SAME star, far from the event (a real non-event light curve).
    Using the same stars for both classes stops the model from learning 'OGLE = lensing'."""
    from cosmic_core import read_ogle_phot
    rng = np.random.default_rng(seed)
    tab = read_table3(table3_path).set_index("name")
    files = sorted(glob.glob(os.path.join(phot_dir, "**", "*.dat"), recursive=True))
    files = [f for f in files if os.path.basename(f).replace(".dat", "") in tab.index]
    if len(files) > max_events:
        files = list(rng.choice(files, max_events, replace=False))
    items = []
    for f in files:
        name = os.path.basename(f).replace(".dat", "")
        try:
            t, m = read_ogle_phot(f)
        except Exception:
            continue
        r = tab.loc[name]
        t0, tE = r.t0_best, r.tE_best
        if not np.isfinite(t0) or not np.isfinite(tE):
            continue
        for a, b in seasons(t):
            ts, ms = t[a:b], m[a:b]
            if len(ts) < min_points:
                continue
            contains = ts.min() - 0.5 * tE < t0 < ts.max() + 0.5 * tE
            far = (t0 < ts.min() - 3 * tE - 30) or (t0 > ts.max() + 3 * tE + 30)
            n_during = np.sum(np.abs(ts - t0) < tE)
            if contains and n_during >= 8:
                items.append(dict(t=ts - ts.min(), v=ms, kind="mag", label=2, group=f"ogle_{name}", source="OGLE",
                                  split=split_of(name), name=name, tE=float(tE), t0_rel=float(t0 - ts.min()),
                                  t_abs=ts, ews=r.ews_id))
            elif far:
                items.append(dict(t=ts - ts.min(), v=ms, kind="mag", label=0, group=f"ogle_{name}", source="OGLE",
                                  split=split_of(name), name=name, tE=float(tE)))
    # keep at most ONE random baseline season per star, so 'Other' doesn't swamp the data
    lens = [it for it in items if it["label"] == 2]
    base = {}
    for it in items:
        if it["label"] == 0:
            base.setdefault(it["group"], []).append(it)
    chosen = [v[int(rng.integers(len(v)))] for v in base.values()]
    return lens + chosen

In [ ]:
sys.path.insert(0, SAVE_DIR)
import importlib, cosmic_core, cosmic_data
importlib.reload(cosmic_core); importlib.reload(cosmic_data)
from cosmic_core import *
from cosmic_data import *

## 3. Load and merge the three datasets into one

In [ ]:
t0_ = time.time()
kep = load_kepler(KEP_TRAIN, KEP_TEST, seed=SEED)
ogle = load_ogle(PHOT_DIR, TABLE3, max_events=MAX_OGLE_EVENTS, seed=SEED)
ast = load_alcdef(ALC_DIR, max_sessions=MAX_ASTEROID_SESSIONS, seed=SEED)
items = kep + ogle + ast
print(f"Loaded {len(items):,} real light curves in {time.time() - t0_:.0f}s")

meta = pd.DataFrame([{k: it.get(k) for k in ["label", "source", "split", "group"]} for it in items])
meta["class"] = meta.label.map(dict(enumerate(SHORT)))
summary = meta.pivot_table(index=["class", "source"], columns="split", values="group", aggfunc="count", fill_value=0)
summary["total"] = summary.sum(axis=1)
display(summary)
print("No object appears in two splits:", meta.groupby("group").split.nunique().max() == 1)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 6))
picks = [(1, "Kepler"), (0, "Kepler"), (2, "OGLE"), (0, "OGLE"), (3, "ALCDEF-ground"), (3, "ALCDEF-TESS")]
rng = np.random.default_rng(1)
for ax, (lab, src) in zip(axes.ravel(), picks + picks[:2]):
    cands = [it for it in items if it["label"] == lab and it["source"] == src and it["split"] == "train"]
    it = cands[rng.integers(len(cands))]
    y = it["v"] if it["kind"] == "flux" else -it["v"]
    ax.plot(it["t"] * (24 if src.startswith("ALCDEF") else 1), y, ".", ms=1.5, color=COLORS[SHORT[lab]])
    ax.set_title(f"{SHORT[lab]} · {src}", fontsize=10); ax.set_xlabel("hours" if src.startswith("ALCDEF") else "days")
plt.suptitle("Real light curves from the merged dataset (brightness ↑)", fontsize=13); plt.tight_layout(); plt.show()

### 📊 Exploring the merged dataset
Before training, we look at the data: how many curves each class has, how long they last, and how many points each has.
**The duration histogram is important:** it shows each phenomenon lives on its own timescale. That's real physics, but also a possible shortcut, which we test later.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(16, 9))
src_order = sorted(meta.source.unique())
counts_cs = meta.groupby(["class", "source"]).size().unstack(fill_value=0).reindex(SHORT)
bottom = np.zeros(4)
for s_, col in zip(src_order, ["#0ea5e9", "#a78bfa", "#f59e0b", "#fb7185", "#22c55e"]):
    vals = counts_cs.get(s_, pd.Series(0, index=SHORT)).values
    ax[0, 0].bar(SHORT, vals, bottom=bottom, label=s_, color=col); bottom += vals
for i, v_ in enumerate(bottom): ax[0, 0].text(i, v_, f"{int(v_):,}", ha="center", va="bottom")
ax[0, 0].set_title("1. Light curves per class, coloured by telescope/source"); ax[0, 0].legend()

dur = np.array([np.log10(max(it["t"].max() - it["t"].min(), 1e-3)) for it in items])
npts = np.array([len(it["t"]) for it in items])
labs = meta.label.values
for k, name in enumerate(SHORT):
    ax[0, 1].hist(dur[labs == k], bins=60, alpha=0.65, color=COLORS[name], label=name, density=True)
    ax[1, 0].hist(np.log10(npts[labs == k]), bins=50, alpha=0.65, color=COLORS[name], label=name, density=True)
ax[0, 1].set_xlabel("log10(duration in days)  [−1 = 2.4 h, 0 = 1 day, 2 = 100 days]"); ax[0, 1].set_title("2. How long each light curve lasts"); ax[0, 1].legend()
ax[1, 0].set_xlabel("log10(number of measurements)"); ax[1, 0].set_title("3. Measurements per light curve"); ax[1, 0].legend()
sp = meta.groupby(["split", "class"]).size().unstack(fill_value=0)[SHORT]
sp.plot(kind="bar", ax=ax[1, 1], color=[COLORS[c] for c in SHORT], rot=0)
ax[1, 1].set_yscale("log"); ax[1, 1].set_title("4. Train / validation / test split (log scale)")
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, "eda_overview.png"), dpi=130); plt.show()

In [ ]:
# Lensing timescales, and asteroid brightness changes: physics quantities in the raw data
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
tEs = np.array([it["tE"] for it in ogle if it["label"] == 2])
ax[0].hist(np.log10(tEs), bins=50, color=COLORS["Lensing"]); ax[0].axvline(2, color="red", ls="--", label="tE = 100 days")
ax[0].set_xlabel("log10 tE (days)"); ax[0].set_title("Einstein timescale of OGLE events"); ax[0].legend()
amps = [np.percentile(it["v"], 95) - np.percentile(it["v"], 5) for it in ast]
ax[1].hist(np.clip(amps, 0, 1), bins=50, color=COLORS["Asteroid"]); ax[1].set_xlabel("amplitude (magnitudes)"); ax[1].set_title("Asteroid brightness variation per session")
pk = [np.percentile(-it["v"], 99) - np.median(-it["v"]) for it in ogle]
ax[2].hist([np.clip([p_ for p_, it in zip(pk, ogle) if it["label"] == lab], 0, 3) for lab in (0, 2)], bins=40,
           color=[COLORS["Other"], COLORS["Lensing"]], label=["normal season", "lensing season"], stacked=False)
ax[2].set_xlabel("brightening above median (mag)"); ax[2].set_title("OGLE: lensing vs normal seasons, same stars"); ax[2].legend()
plt.tight_layout(); plt.show()

## 4. Pre-processing and augmentation
**Augmentation (training data only):** we randomly cut gaps, add extra noise and flip time. This happens for **every** source, so things like gaps can't reveal which telescope a curve came from.
There are only 37 training planets, so each one gets extra augmented copies. For planets we also shift the curve in time (a transit can happen at any moment).

In [ ]:
def augment(t, v, kind, rng, is_planet=False):
    t, v = t.copy(), v.astype(float).copy()
    if is_planet:                                         # circular time shift (uniform Kepler sampling)
        v = np.roll(v, rng.integers(len(v)))
    if rng.random() < 0.5:                                # time reversal: all four phenomena are ~symmetric
        t = t.max() - t[::-1]; v = v[::-1]
    if rng.random() < 0.7:                                # random gaps
        keep = np.ones(len(t), bool)
        for _ in range(rng.integers(1, 4)):
            w = rng.uniform(0.03, 0.15) * (t.max() - t.min()); c = rng.uniform(t.min(), t.max())
            keep &= ~((t > c - w / 2) & (t < c + w / 2))
        if keep.sum() > 20:
            t, v = t[keep], v[keep]
    noise = np.median(np.abs(np.diff(v))) / 1.4
    v = v + rng.normal(0, rng.uniform(0, 1.0) * noise, len(v))
    return t, v

def build_arrays(items, split, augment_train=True, seed=SEED):
    rng = np.random.default_rng(seed)
    X, D, Y, S, G = [], [], [], [], []
    for it in items:
        if it["split"] != split:
            continue
        versions = [(it["t"], it["v"])]
        if split == "train" and augment_train:
            n_aug = PLANET_AUG_COPIES if it["label"] == 1 else 1
            versions += [augment(it["t"], it["v"], it["kind"], rng, is_planet=it["label"] == 1) for _ in range(n_aug)]
        for k, (t, v) in enumerate(versions):
            out = preprocess(t, v, it["kind"], seed=int(rng.integers(1e9)))
            if out is None:
                continue
            X.append(out[0]); D.append(out[1]); Y.append(it["label"]); S.append(it["source"]); G.append(it["group"])
    return np.stack(X), np.array(D, np.float32), np.array(Y), np.array(S), np.array(G)

t0_ = time.time()
Xtr, Dtr, ytr, Str, Gtr = build_arrays(items, "train")
Xva, Dva, yva, Sva, Gva = build_arrays(items, "val")
Xte, Dte, yte, Ste, Gte = build_arrays(items, "test")
print(f"Pre-processed in {time.time() - t0_:.0f}s → train {Xtr.shape}, val {Xva.shape}, test {Xte.shape}")
print("Train class counts (after augmentation):", dict(zip(SHORT, np.bincount(ytr, minlength=4))))
print("Test class counts (real, untouched):    ", dict(zip(SHORT, np.bincount(yte, minlength=4))))

### 📊 Pre-processing, step by step
One real example per class: raw data → the two channels the CNN actually sees.

In [ ]:
fig, ax = plt.subplots(4, 3, figsize=(17, 11))
rng_ = np.random.default_rng(3)
for r, lab in enumerate(range(4)):
    c = [it for it in items if it["label"] == lab and it["split"] == "train"]
    it = c[rng_.integers(len(c))]
    X_, d_ = preprocess(it["t"], it["v"], it["kind"])
    raw = it["v"] if it["kind"] == "flux" else -it["v"]
    ax[r, 0].plot(it["t"], raw, ".", ms=1.5, color=COLORS[SHORT[lab]]); ax[r, 0].set_title(f"{SHORT[lab]} · {it['source']}: RAW ({len(it['t'])} points)")
    ax[r, 1].plot(X_[:, 0], color="#334155", lw=0.8); ax[r, 1].set_title(f"Channel A: binned + normalised (duration 10^{d_:.2f} d)")
    ax[r, 2].plot(X_[:, 1], color=COLORS[SHORT[lab]], lw=0.7); ax[r, 2].set_title("Channel B: high-pass (fast changes)")
ax[3, 0].set_xlabel("days"); ax[3, 1].set_xlabel("bin (of 1,024)"); ax[3, 2].set_xlabel("bin (of 1,024)")
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, "preprocessing_steps.png"), dpi=120); plt.show()

In [ ]:
# Augmentation: one real planet and its augmented copies
pl = [it for it in items if it["label"] == 1 and it["split"] == "train"][0]
fig, ax = plt.subplots(1, 4, figsize=(18, 3))
ax[0].plot(pl["t"], pl["v"], ".", ms=1, color=COLORS["Planet"]); ax[0].set_title("Original planet light curve")
rng_ = np.random.default_rng(7)
for k in range(1, 4):
    ta, va = augment(pl["t"], pl["v"], "flux", rng_, is_planet=True)
    ax[k].plot(ta, va, ".", ms=1, color="#0f766e"); ax[k].set_title(f"Augmented copy {k} (shift / flip / gaps / noise)")
plt.tight_layout(); plt.show()

In [ ]:
# Histograms of the model inputs per class: do the classes look different after pre-processing?
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
for k, name in enumerate(SHORT):
    m_ = yva == k
    if m_.sum() == 0: continue
    ax[0].hist(Xva[m_, :, 0].std(1), bins=50, alpha=0.6, color=COLORS[name], label=name, density=True)
    ax[1].hist(Xva[m_, :, 1].min(1), bins=50, alpha=0.6, color=COLORS[name], label=name, density=True)
    ax[2].hist(Dva[m_], bins=40, alpha=0.6, color=COLORS[name], label=name, density=True)
ax[0].set_title("Spread of channel A (big = strong slow variation)"); ax[1].set_title("Deepest dip in channel B (planets → very negative)")
ax[2].set_title("Duration input log10(days)"); [a.legend(fontsize=8) for a in ax]
plt.tight_layout(); plt.show()

## 5. Baseline: Random Forest on hand-made features
Before deep learning, we try a classic method. If the CNN can't beat this, it isn't worth it.

In [ ]:
def features(X, D):
    a, b = X[..., 0], X[..., 1]
    f = [D[:, None]]
    for z in (a, b):
        f += [z.mean(1, keepdims=True), z.std(1, keepdims=True), np.percentile(z, [1, 5, 50, 95, 99], axis=1).T,
              ((z - z.mean(1, keepdims=True)) ** 3).mean(1, keepdims=True) / (z.std(1, keepdims=True) ** 3 + 1e-9),
              (z < -2).mean(1, keepdims=True), (z > 2).mean(1, keepdims=True)]
        spec = np.abs(np.fft.rfft(z, axis=1))[:, 1:65]
        f += [np.log1p(spec[:, ::4])]
    return np.concatenate(f, axis=1)

Ftr, Fva, Fte = features(Xtr, Dtr), features(Xva, Dva), features(Xte, Dte)
rf = RandomForestClassifier(n_estimators=400, class_weight="balanced_subsample", min_samples_leaf=2, n_jobs=-1, random_state=SEED)
rf.fit(Ftr, ytr)
p_rf = rf.predict_proba(Fte)
print("Random Forest – test macro-F1:", round(f1_score(yte, p_rf.argmax(1), average="macro"), 3))

feat_names = ["duration"] + [f"{ch}:{n}" for ch in ("A", "B") for n in
              ["mean", "std", "p1", "p5", "p50", "p95", "p99", "skew", "frac<-2", "frac>2"] + [f"fft{i}" for i in range(1, 65, 4)]]
imp = pd.Series(rf.feature_importances_, index=feat_names).sort_values()[-15:]
imp.plot(kind="barh", figsize=(8, 5), color="#64748b", title="Random Forest: 15 most important features"); plt.tight_layout(); plt.show()

## 6. The deep-learning model: a 1D residual CNN
- The **convolution filters** slide along the light curve and learn shapes: a dip, a bump, a wave.
- **Residual connections** (as in ResNet) let us stack many layers without training problems.
- **Global pooling** makes the model care *what* shape appears, not *where* it appears.
- The duration number joins the network just before the final decision.

**Class weights** make mistakes on rare classes (planets!) cost more.

In [ ]:
def res_block(x, f, k=7, stride=1):
    sc = x
    x = layers.Conv1D(f, k, strides=stride, padding="same", use_bias=False)(x); x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Conv1D(f, k, padding="same", use_bias=False)(x); x = layers.BatchNormalization()(x)
    if sc.shape[-1] != f or stride != 1:
        sc = layers.Conv1D(f, 1, strides=stride, use_bias=False)(sc); sc = layers.BatchNormalization()(sc)
    return layers.Activation("relu")(layers.Add()([x, sc]))

def build_model(use_duration=True):
    xin = layers.Input((L, 2), name="curve")
    x = layers.Conv1D(32, 9, strides=2, padding="same")(xin)           # 1024 → 512
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(2)(x)                                        # 512 → 256
    for f, s in [(32, 1), (64, 2), (64, 1), (128, 2), (128, 1)]:         # 256 → 64
        x = res_block(x, f, stride=s)
    x = layers.Concatenate()([layers.GlobalAveragePooling1D()(x), layers.GlobalMaxPooling1D()(x)])
    inputs = [xin]
    if use_duration:
        din = layers.Input((1,), name="duration"); inputs.append(din)
        x = layers.Concatenate()([x, layers.Dense(16, activation="relu")(din)])
    x = layers.Dropout(0.4)(x); x = layers.Dense(128, activation="relu")(x); x = layers.Dropout(0.3)(x)
    out = layers.Dense(4, activation="softmax")(x)
    m = keras.Model(inputs, out, name="CosmicDetector" if use_duration else "CosmicDetector_noDuration")
    m.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

counts = np.bincount(ytr, minlength=4)
class_weight = {k: float(np.sqrt(counts.max() / counts[k])) for k in range(4)}
print("class weights:", {SHORT[k]: round(v, 2) for k, v in class_weight.items()})

class MacroF1(keras.callbacks.Callback):
    """Track macro-F1 on validation (fair to all classes, unlike accuracy)."""
    def __init__(self, inputs, y): super().__init__(); self.inputs, self.y = inputs, y
    def on_epoch_end(self, epoch, logs=None):
        p = self.model.predict(self.inputs, verbose=0, batch_size=512).argmax(1)
        logs["val_macro_f1"] = f1_score(self.y, p, average="macro")
        print(f"   epoch {epoch + 1}: val macro-F1 = {logs['val_macro_f1']:.3f}")

def train(use_duration=True):
    tf.keras.utils.set_random_seed(SEED)
    m = build_model(use_duration)
    inp = lambda X, D: {"curve": X, "duration": D} if use_duration else {"curve": X}
    h = m.fit(inp(Xtr, Dtr), ytr, validation_data=(inp(Xva, Dva), yva), epochs=EPOCHS, batch_size=BATCH,
              class_weight=class_weight, verbose=0,
              callbacks=[MacroF1(inp(Xva, Dva), yva),
                         keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max", patience=7, restore_best_weights=True),
                         keras.callbacks.ReduceLROnPlateau(monitor="val_macro_f1", mode="max", factor=0.5, patience=3)])
    return m, h, inp

model, hist, inp = train(use_duration=True)
model.summary(line_length=100, print_fn=lambda s: None)
print("Trainable parameters:", f"{model.count_params():,}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="validation"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(hist.history["val_macro_f1"], color="tab:green"); ax[1].set_title("Validation macro-F1"); [a.set_xlabel("epoch") for a in ax]
plt.tight_layout(); plt.show()

## 7. Honest evaluation on the untouched test set
**Macro-F1** averages the F1 score of the 4 classes equally, so the rare planet class counts as much as the big classes. **Balanced accuracy** is the average recall per class.

In [ ]:
p_cnn = model.predict(inp(Xte, Dte), verbose=0, batch_size=512)
p_val = model.predict(inp(Xva, Dva), verbose=0, batch_size=512)

# Planets are so rare that "highest probability wins" gives many false alarms.
# We choose a stricter planet threshold on the VALIDATION set (never on the test set).
from sklearn.metrics import precision_recall_curve
pr_, rc_, th_ = precision_recall_curve(yva == 1, p_val[:, 1])
PLANET_THRESHOLD = float(th_[np.nanargmax(2 * pr_[:-1] * rc_[:-1] / (pr_[:-1] + rc_[:-1] + 1e-9))]) if len(th_) else 0.5
def decide(p, tau=PLANET_THRESHOLD):
    pred = p.argmax(1)
    weak = (pred == 1) & (p[:, 1] < tau)
    q = p.copy(); q[:, 1] = -1
    pred[weak] = q[weak].argmax(1)
    return pred
pred_argmax = p_cnn.argmax(1)
pred = decide(p_cnn)
print(f"Planet threshold chosen on validation: {PLANET_THRESHOLD:.3f}")
print("Kepler planets with plain argmax: found", int(((pred_argmax == 1) & (yte == 1)).sum()), "of", int((yte == 1).sum()),
      "| false alarms", int(((pred_argmax == 1) & (yte != 1)).sum()))
print("Kepler planets with tuned threshold: found", int(((pred == 1) & (yte == 1)).sum()), "of", int((yte == 1).sum()),
      "| false alarms", int(((pred == 1) & (yte != 1)).sum()))
print(classification_report(yte, pred, target_names=SHORT, digits=3, zero_division=0))
res = {"CNN": {"macro_f1": f1_score(yte, pred, average="macro"), "balanced_acc": balanced_accuracy_score(yte, pred)},
       "CNN (plain argmax)": {"macro_f1": f1_score(yte, pred_argmax, average="macro"), "balanced_acc": balanced_accuracy_score(yte, pred_argmax)},
       "Random Forest": {"macro_f1": f1_score(yte, p_rf.argmax(1), average="macro"), "balanced_acc": balanced_accuracy_score(yte, p_rf.argmax(1))}}

fig, ax = plt.subplots(1, 2, figsize=(14, 5.2))
cm = confusion_matrix(yte, pred, labels=range(4))
cmn = cm / cm.sum(1, keepdims=True)
ax[0].imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for i in range(4):
    for j in range(4):
        ax[0].text(j, i, f"{cm[i, j]}\n{cmn[i, j]:.0%}", ha="center", va="center", color="white" if cmn[i, j] > 0.5 else "black", fontsize=10)
ax[0].set_xticks(range(4), SHORT); ax[0].set_yticks(range(4), SHORT); ax[0].set_xlabel("predicted"); ax[0].set_ylabel("true"); ax[0].grid(False)
ax[0].set_title("Confusion matrix (test set): count and % of true class")
for k, name in enumerate(SHORT):
    ax[1].bar(k - 0.2, average_precision_score(yte == k, p_rf[:, k]), 0.4, color="#cbd5e1", label="Random Forest" if k == 0 else None)
    ax[1].bar(k + 0.2, average_precision_score(yte == k, p_cnn[:, k]), 0.4, color=COLORS[name], label="CNN" if k == 0 else None)
ax[1].set_xticks(range(4), SHORT); ax[1].set_ylim(0, 1.05); ax[1].set_title("PR-AUC per class (1 = perfect)"); ax[1].legend()
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=130); plt.show()

### 7a. The hard questions, asked *inside* each telescope
Different classes mostly come from different telescopes, so a high overall score could just mean "the model recognises the telescope". The real tests are **inside** one telescope:
- **Kepler:** can it find the 5 planets among 570 test stars?
- **OGLE:** can it tell a lensing season from a normal season **of the same kind of star, from the same telescope**?

In [ ]:
def within(src, pos, neg=0):
    m = (Ste == src) & np.isin(yte, [pos, neg])
    yt, pp = (yte[m] == pos).astype(int), p_cnn[m][:, pos]
    hit = pred[m] == pos
    return {"source": src, "positives": int(yt.sum()), "negatives": int((1 - yt).sum()),
            "found": f"{int((hit & (yt == 1)).sum())}/{int(yt.sum())}", "false alarms": int((hit & (yt == 0)).sum()),
            "PR-AUC": round(average_precision_score(yt, pp), 3) if yt.sum() else None}

w = pd.DataFrame([within("Kepler", 1), within("OGLE", 2)], index=["Planets in Kepler", "Lensing in OGLE"])
display(w)
print("Asteroid recall (all asteroid test sessions):", round((pred[yte == 3] == 3).mean(), 3))
res["within"] = w.to_dict(orient="index")

In [ ]:
# Accuracy per source, and how confident the model is per class
fig, ax = plt.subplots(1, 2, figsize=(16, 4.2))
srcs = sorted(set(Ste)); acc = [(pred[Ste == s_] == yte[Ste == s_]).mean() for s_ in srcs]
ax[0].bar(srcs, acc, color="#38bdf8"); [ax[0].text(i, a + 0.01, f"{a:.1%}\nn={int((Ste == s_).sum())}", ha="center") for i, (a, s_) in enumerate(zip(acc, srcs))]
ax[0].set_ylim(0, 1.15); ax[0].set_title("Test accuracy per data source")
for k, name in enumerate(SHORT):
    if (yte == k).any():
        ax[1].hist(p_cnn[yte == k, k], bins=25, alpha=0.6, color=COLORS[name], label=f"true {name}", density=True)
ax[1].set_xlabel("probability the model gives to the TRUE class"); ax[1].set_title("Confidence on the correct class (right = good)"); ax[1].legend()
plt.tight_layout(); plt.show()

### 🔍 Where does the model go wrong?
Showing mistakes is part of honest evaluation. Below are real test curves the model got **wrong**.

In [ ]:
wrong = np.where(pred != yte)[0]
print(f"{len(wrong)} wrong out of {len(yte)} test curves")
if len(wrong):
    sel = np.random.default_rng(0).choice(wrong, min(8, len(wrong)), replace=False)
    fig, axes = plt.subplots(2, 4, figsize=(18, 6))
    for ax_, i in zip(axes.ravel(), sel):
        ax_.plot(Xte[i, :, 0], color="#334155", lw=0.7); ax_.plot(Xte[i, :, 1], color=COLORS[SHORT[pred[i]]], lw=0.5, alpha=0.7)
        ax_.set_title(f"true {SHORT[yte[i]]} → predicted {SHORT[pred[i]]}\n{Ste[i]} · p={p_cnn[i].max():.2f}", fontsize=9)
    plt.tight_layout(); plt.show()

### 🧠 What is the CNN looking at? (saliency maps)
We measure how much each part of the light curve changes the model's decision (the gradient of the output with respect to the input). Red marks the parts that matter most.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(16, 10))
for k, ax_ in enumerate(axes):
    idx = np.where((yte == k) & (pred == k))[0]
    if not len(idx): continue
    i = idx[np.argmax(p_cnn[idx, k])]
    x_ = tf.convert_to_tensor(Xte[i:i + 1]); d_ = tf.convert_to_tensor(Dte[i:i + 1, None])
    with tf.GradientTape() as tape:
        tape.watch(x_); out = model({"curve": x_, "duration": d_}, training=False)[0, k]
    g = np.abs(tape.gradient(out, x_).numpy()[0]).sum(1); g = np.convolve(g, np.ones(9) / 9, "same")
    ax_.plot(Xte[i, :, 0], color="black", lw=0.7)
    ax2 = ax_.twinx(); ax2.fill_between(range(L), 0, g, color="red", alpha=0.3); ax2.set_yticks([])
    ax_.set_title(f"{SHORT[k]} ({Ste[i]}): red = where the CNN looks", fontsize=10)
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, "saliency.png"), dpi=120); plt.show()

### 7b. Shortcut check: does the model just read the duration?
We train the same network **without** the duration number. If performance collapses, the model was mostly reading timescales and not shapes. That would be important to admit.

In [ ]:
if RUN_ABLATION:
    model_nd, _, inp_nd = train(use_duration=False)
    p_nd = model_nd.predict(inp_nd(Xte, Dte), verbose=0, batch_size=512)
    res["CNN without duration"] = {"macro_f1": f1_score(yte, p_nd.argmax(1), average="macro"),
                                   "balanced_acc": balanced_accuracy_score(yte, p_nd.argmax(1))}
    print(classification_report(yte, p_nd.argmax(1), target_names=SHORT, digits=3, zero_division=0))
display(pd.DataFrame({k: v for k, v in res.items() if k != "within"}).T.round(3))

## 8. Physics stage: from "lensing" to "black-hole candidate"
A lensing event alone doesn't tell us what the lens is. Astronomers fit the **Paczyński model** to measure the **Einstein timescale tE**: how long the lens takes to cross its own "Einstein radius".
Heavier lenses give longer events (tE ∝ √mass), so **very long events (tE > 100 days)** are the classic **black-hole candidates**. That's how the first confirmed isolated stellar-mass black hole, OGLE-2011-BLG-0462, was flagged.

**Check:** we fit tE ourselves on test events and compare with OGLE's published values.

In [ ]:
test_lenses = [it for it in ogle if it["label"] == 2 and it["split"] == "test"]
fits = []
for it in test_lenses[:300]:
    t_full, m_full = read_ogle_phot(os.path.join(PHOT_DIR, "phot", it["name"] + ".dat")) \
        if os.path.exists(os.path.join(PHOT_DIR, "phot", it["name"] + ".dat")) else (it["t"], it["v"])
    r = fit_lens(t_full, m_full)          # fit on ALL seasons: the extra baseline makes tE more accurate
    if r:
        fits.append({"name": it["name"], "tE_ours": r["tE_days"], "tE_OGLE": it["tE"]})
fits = pd.DataFrame(fits)
good = fits[(fits.tE_OGLE > 0) & (fits.tE_ours > 0)]
ratio = good.tE_ours / good.tE_OGLE
print(f"Fitted {len(good)} test events. Our tE is within 20% of OGLE's for {np.mean(np.abs(ratio - 1) < 0.2):.0%} of them "
      f"(median ratio {np.median(ratio):.2f}).")
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].loglog(good.tE_OGLE, good.tE_ours, ".", alpha=0.6, color=COLORS["Lensing"]); lim = [1, 1000]
ax[0].plot(lim, lim, "k--", lw=1); ax[0].set_xlabel("tE published by OGLE (days)"); ax[0].set_ylabel("tE fitted by us (days)")
ax[0].set_title("Our physics fit vs. the professional result")
all_tE = np.array([it["tE"] for it in ogle if it["label"] == 2])
ax[1].hist(np.log10(all_tE[all_tE > 0]), bins=50, color=COLORS["Lensing"]); ax[1].axvline(2, color="red", ls="--", label="tE = 100 d (black-hole candidates →)")
ax[1].set_xlabel("log10 tE (days)"); ax[1].set_title(f"{(all_tE > 100).sum()} of {len(all_tE)} events are black-hole candidates"); ax[1].legend()
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, "lens_physics.png"), dpi=130); plt.show()
res["tE_within_20pct"] = float(np.mean(np.abs(ratio - 1) < 0.2)); res["bh_candidates"] = int((all_tE > 100).sum())

## 9. "Future data" test: events the model has never seen
The OGLE archive covers 2010–2017. Any OGLE event from **2018 or later** is a true test of the future.
Download events from the OGLE Early Warning System (https://ogle.astrouw.edu.pl/ogle4/ews/ews.html → pick a year → an event → the download link) and put the `.tar` / `.tar.gz` files or `phot.dat` files in `MyDrive/CosmicData/ews/`.

In [ ]:
EWS_DIR = os.path.join(DATA_DIR, "ews"); os.makedirs(EWS_DIR, exist_ok=True)
for f in os.listdir(EWS_DIR):
    if f.endswith((".tar", ".tar.gz", ".tgz")):
        with tarfile.open(os.path.join(EWS_DIR, f)) as tf_:
            tf_.extractall(EWS_DIR)

def classify(t, v, kind):
    X, d = preprocess(t, v, kind)
    p = model.predict({"curve": X[None], "duration": np.array([d])}, verbose=0)[0]
    return p, X, d

future = []
for root, _, fs in os.walk(EWS_DIR):
    if "phot.dat" in fs:
        t, m = read_ogle_phot(os.path.join(root, "phot.dat"))
        pars = open(os.path.join(root, "params.dat")).read() if "params.dat" in fs else ""
        tmax = [float(l.split()[1]) for l in pars.splitlines() if l.strip().startswith("Tmax")]
        tau = [float(l.split()[1]) for l in pars.splitlines() if l.strip().startswith("tau")]
        fit = fit_lens(t, m)                      # physics fit on the full multi-year light curve
        if tmax:   # classify the observing season containing the event, like in training
            for a, b in seasons(t):
                if t[a] - 30 < tmax[0] < t[b - 1] + 30:
                    t, m = t[a:b], m[a:b]; break
        p, _, _ = classify(t - t.min(), m, "mag")
        future.append({"event": os.path.basename(root), "prediction": SHORT[p.argmax()], "P(lensing)": round(float(p[2]), 3),
                       "tE ours": round(fit["tE_days"], 1) if fit else None, "tE OGLE": tau[0] if tau else None,
                       "black-hole candidate": bool(fit and fit["tE_days"] > 100)})
display(pd.DataFrame(future)) if future else print("No future events found in", EWS_DIR)

## 10. Save the model

In [ ]:
model.save(os.path.join(SAVE_DIR, "cosmic_detector.keras"))
def _clean(o):
    if isinstance(o, dict): return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, (np.floating, np.integer)): return o.item()
    return o
json.dump(_clean({"results": res, "planet_threshold": PLANET_THRESHOLD, "test_counts": dict(zip(SHORT, np.bincount(yte, minlength=4).tolist())),
                  "train_duration_range": [float(Dtr.min()), float(Dtr.max())]}),
          open(os.path.join(SAVE_DIR, "cosmic_results.json"), "w"), indent=2)
# a small gallery of real test curves for the app
gallery = []
for lab in range(4):
    for src in sorted(set(it["source"] for it in items)):
        c = [it for it in items if it["split"] == "test" and it["label"] == lab and it["source"] == src]
        for it in c[:6]:
            gallery.append({"label": lab, "source": src, "name": it.get("name", it["group"]), "kind": it["kind"],
                            "t": np.asarray(it["t"]).tolist(), "v": np.asarray(it["v"]).tolist()})
json.dump(gallery, open(os.path.join(SAVE_DIR, "gallery.json"), "w"))
print("✅ Saved model, results and a gallery of", len(gallery), "real test curves to", SAVE_DIR)

## 11. 🖥️ The Cosmic Detector app
Run the cell below. A link appears (`https://….gradio.live`); open it on any device. You can:
- pick a **real test light curve** the model has never seen, or
- **upload your own** file: an OGLE `phot.dat`, or any text/CSV file with *time (days)* and *magnitude or flux* columns.

The app shows the model's verdict, the probability of each class, and the **physics measurements** for that class (Einstein time for lensing → black-hole check, rotation period for asteroids, orbital period for planets).

In [ ]:
import gradio as gr, inspect, io, tempfile
import matplotlib
from matplotlib.figure import Figure

APP_MODEL = keras.models.load_model(os.path.join(SAVE_DIR, "cosmic_detector.keras"))
APP_RES = json.load(open(os.path.join(SAVE_DIR, "cosmic_results.json")))
GALLERY = json.load(open(os.path.join(SAVE_DIR, "gallery.json")))
DUR_RANGE = APP_RES["train_duration_range"]
ICON = {"Other": "⭕", "Planet": "🪐", "Lensing": "🕳️", "Asteroid": "☄️"}
BG, PANEL, INK, MUTED = "#0b1020", "#131b33", "#e6e9f2", "#8b93a7"

def dark_fig(h=3.2):
    fig = Figure(figsize=(9, h), facecolor=PANEL)
    return fig

def style(ax):
    ax.set_facecolor(PANEL); ax.tick_params(colors=MUTED, labelsize=8)
    for s in ax.spines.values(): s.set_color("#2a3352")
    ax.grid(alpha=0.15); ax.title.set_color(INK); ax.xaxis.label.set_color(MUTED); ax.yaxis.label.set_color(MUTED)

def prob_bars(p):
    rows = ""
    for k in np.argsort(-p):
        name = SHORT[k]; pct = 100 * p[k]
        rows += (f'<div class="pb-row"><span class="pb-name">{ICON[name]} {name}</span>'
                 f'<div class="pb-track"><div class="pb-fill" style="width:{pct:.1f}%;background:{COLORS[name]}"></div></div>'
                 f'<span class="pb-val">{pct:.1f}%</span></div>')
    return rows

def analyse(t, v, kind, title="Your light curve", truth=None, full=None):
    t = np.asarray(t, float); v = np.asarray(v, float)
    out = preprocess(t - t.min(), v, kind)
    if out is None:
        return "<div class='card'>❌ Not enough valid points (need at least 10).</div>", None, ""
    X, d = out
    p = APP_MODEL.predict({"curve": X[None], "duration": np.array([d])}, verbose=0)[0]
    k = int(decide(p[None], APP_RES.get("planet_threshold", 0.5))[0]); name = SHORT[k]
    warn = ""
    if not (DUR_RANGE[0] - 0.3 <= d <= DUR_RANGE[1] + 0.3):
        warn = (f"<div class='warn'>⚠️ This curve lasts {10**d:.2f} days, outside the training range "
                f"({10**DUR_RANGE[0]:.2f}–{10**DUR_RANGE[1]:.0f} d). Treat the verdict with caution.</div>")
    if p[k] < 0.6:
        warn += "<div class='warn'>🤔 Low confidence: the model is unsure between classes.</div>"
    truth_html = ""
    if truth is not None:
        ok = truth == k
        truth_html = (f"<div class='truth {'ok' if ok else 'bad'}'>True class: {ICON[SHORT[truth]]} {SHORT[truth]} · "
                      f"{'✅ correct' if ok else '❌ model was wrong'}</div>")
    verdict = (f"<div class='card'><div class='eyebrow'>VERDICT</div>"
               f"<div class='verdict' style='color:{COLORS[name]}'>{ICON[name]} {CLASSES[k]}</div>"
               f"<div class='conf'>confidence {100*p[k]:.1f}%</div>{truth_html}{warn}"
               f"<div class='pb'>{prob_bars(p)}</div></div>")

    # physics stage
    phys = "<div class='card'><div class='eyebrow'>PHYSICS MEASUREMENTS</div>"
    fig = dark_fig(5.6); ax1 = fig.add_subplot(2, 1, 1); ax2 = fig.add_subplot(2, 1, 2)
    bright = -v if kind == "mag" else v
    hours = (t.max() - t.min()) < 3
    tx = (t - t.min()) * (24 if hours else 1)
    ax1.plot(tx, bright, ".", ms=2.5, color=COLORS[name], alpha=0.8)
    ax1.set_title(title); ax1.set_xlabel("hours" if hours else "days"); ax1.set_ylabel("brightness ↑" if kind == "mag" else "flux")
    if name == "Lensing" and kind == "mag":
        fit = fit_lens(*full) if full is not None else fit_lens(t, v)
        if fit:
            tt = np.linspace(t.min(), t.max(), 1500)
            model_flux = paczynski(tt, *fit["params"])
            base_mag = np.median(v)
            ax1.plot((tt - t.min()), -(base_mag - 2.5 * np.log10(model_flux)), color="white", lw=1.6, label="Paczyński fit")
            ax1.legend(facecolor=PANEL, labelcolor=INK, fontsize=8)
            bh = fit["tE_days"] > 100
            phys += (f"<div class='kv'><span>Einstein timescale tE</span><b>{fit['tE_days']:.1f} days</b></div>"
                     f"<div class='kv'><span>Impact parameter u0</span><b>{fit['u0']:.3f}</b></div>"
                     f"<div class='kv'><span>Peak magnification</span><b>{(fit['u0']**2+2)/(fit['u0']*np.sqrt(fit['u0']**2+4)):.1f}×</b></div>"
                     f"<div class='bh {'yes' if bh else 'no'}'>{'🕳️ BLACK-HOLE CANDIDATE: tE > 100 days means a heavy, dark lens' if bh else 'Ordinary lens most likely (tE < 100 days): probably a star or brown dwarf'}</div>")
    elif name == "Asteroid":
        P = rotation_period_hours(t, v)
        phys += (f"<div class='kv'><span>Rotation period (Lomb-Scargle, double-peaked)</span><b>{P:.2f} h</b></div>" if P else "")
        phys += f"<div class='kv'><span>Amplitude</span><b>{np.percentile(v,95)-np.percentile(v,5):.3f} mag</b></div>"
        phys += "<div class='note'>A single night can be shorter than one rotation, so treat the period as an estimate.</div>"
    elif name == "Planet":
        try:
            P, depth, snr = transit_period_days(t, v)
            phys += (f"<div class='kv'><span>Orbital period (BLS)</span><b>{P:.3f} days</b></div>"
                     f"<div class='kv'><span>Transit signal-to-noise</span><b>{snr:.1f}</b></div>")
        except Exception:
            pass
    else:
        phys += "<div class='note'>No planet, lensing or asteroid signature detected.</div>"
    phys += "</div>"
    ax2.plot(X[:, 0], color="#cbd5e1", lw=0.8, label="channel A (whole curve)")
    ax2.plot(X[:, 1], color=COLORS[name], lw=0.6, alpha=0.8, label="channel B (fast changes)")
    ax2.set_title(f"What the CNN sees: 1,024 bins · log10(duration) = {d:.2f}"); ax2.legend(facecolor=PANEL, labelcolor=INK, fontsize=8, loc="upper right")
    for a in (ax1, ax2): style(a)
    fig.tight_layout()
    return verdict, fig, phys

def gallery_choices(label):
    lab = SHORT.index(label)
    return [f"{i}: {g['source']} · {g['name']}" for i, g in enumerate(GALLERY) if g["label"] == lab]

def run_gallery(choice):
    if not choice:
        return "<div class='card'>Pick a light curve first.</div>", None, ""
    g = GALLERY[int(choice.split(":")[0])]
    return analyse(g["t"], g["v"], g["kind"], title=f"{g['source']} · {g['name']} (real test data)", truth=g["label"])

def run_upload(file, kind):
    if file is None:
        return "<div class='card'>Upload a file first.</div>", None, ""
    path = file if isinstance(file, str) else file.name
    try:
        t, v = read_ogle_phot(path) if kind == "OGLE phot.dat" else read_two_columns(path)
    except Exception as e:
        return f"<div class='card'>❌ Could not read the file: {e}</div>", None, ""
    k = "flux" if kind == "time, flux" else "mag"
    full = None
    if kind == "OGLE phot.dat" and len(t) > 0:       # classify the season with the brightest point, like training
        full = (t, v)
        segs = seasons(t); a, b = max(segs, key=lambda s: -np.min(v[s[0]:s[1]]) if s[1] > s[0] else -99)
        t, v = t[a:b], v[a:b]
    return analyse(t, v, k, title=os.path.basename(path), full=full)

R = APP_RES["results"]
w = R.get("within", {})
def fmt(x): return f"{x:.3f}" if isinstance(x, (int, float)) else str(x)
report_rows = "".join(f"<tr><td>{m}</td><td>{fmt(v.get('macro_f1'))}</td><td>{fmt(v.get('balanced_acc'))}</td></tr>"
                      for m, v in R.items() if isinstance(v, dict) and "macro_f1" in v)
within_rows = "".join(f"<tr><td>{k}</td><td>{v.get('found')}</td><td>{v.get('false alarms')}</td><td>{fmt(v.get('PR-AUC'))}</td></tr>" for k, v in w.items())

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@400;600;700&family=Inter:wght@400;500&display=swap');
.gradio-container {background: radial-gradient(1200px 600px at 10% -10%, #1b2550 0%, #0b1020 55%) !important; font-family: Inter, system-ui, -apple-system, 'Segoe UI', sans-serif; color:#e6e9f2;}
#hero h1 {font-family:'Space Grotesk',Inter,system-ui,-apple-system,'Segoe UI',sans-serif; font-size:42px; letter-spacing:4px; margin:0; background:linear-gradient(90deg,#2dd4bf,#a78bfa,#f59e0b); -webkit-background-clip:text; color:transparent;}
#hero p {color:#8b93a7; margin-top:4px; font-size:15px}
.chips {display:flex; gap:8px; flex-wrap:wrap; margin-top:10px}
.chip {padding:4px 12px; border-radius:999px; font-size:13px; border:1px solid #2a3352; background:#131b33}
.card {background:#131b33; border:1px solid #2a3352; border-radius:16px; padding:18px 20px; margin-bottom:12px}
.eyebrow {font-size:11px; letter-spacing:2px; color:#8b93a7; font-weight:600}
.verdict {font-family:'Space Grotesk',Inter,system-ui,-apple-system,'Segoe UI',sans-serif; font-size:26px; font-weight:700; margin:6px 0 2px}
.conf {color:#8b93a7; font-size:14px; margin-bottom:10px}
.pb-row {display:grid; grid-template-columns:120px 1fr 56px; align-items:center; gap:10px; margin:7px 0; font-size:14px}
.pb-track {height:10px; background:#0b1020; border-radius:99px; overflow:hidden}
.pb-fill {height:100%; border-radius:99px}
label span {color:#e6e9f2 !important}
.pb-val {text-align:right; color:#cbd5e1; font-variant-numeric:tabular-nums}
.kv {display:flex; justify-content:space-between; padding:8px 0; border-bottom:1px dashed #2a3352; font-size:14px}
.kv b {font-family:'Space Grotesk',Inter,system-ui,sans-serif}
.bh {margin-top:12px; padding:10px 12px; border-radius:10px; font-weight:600; font-size:14px}
.bh.yes {background:rgba(167,139,250,.15); color:#c4b5fd; border:1px solid #7c3aed}
.bh.no {background:#0b1020; color:#8b93a7}
.warn {margin-top:8px; padding:8px 10px; border-radius:8px; background:rgba(245,158,11,.12); color:#fbbf24; font-size:13px}
.truth {margin:4px 0 8px; font-size:13px; padding:6px 10px; border-radius:8px}
.truth.ok {background:rgba(45,212,191,.12); color:#5eead4} .truth.bad {background:rgba(248,113,113,.12); color:#fca5a5}
.note {color:#8b93a7; font-size:13px; margin-top:8px}
table.rep {width:100%; border-collapse:collapse; font-size:14px} table.rep td, table.rep th {padding:8px; border-bottom:1px solid #2a3352; text-align:left}
"""
HERO = f"""<div id='hero'><h1>COSMIC DETECTOR</h1>
<p>One deep-learning model · real NASA Kepler, OGLE-IV and ALCDEF light curves · planets, black-hole candidates and asteroids</p>
<div class='chips'><span class='chip' style='color:{COLORS["Planet"]}'>🪐 Exoplanet transits</span>
<span class='chip' style='color:{COLORS["Lensing"]}'>🕳️ Microlensing → black-hole candidates</span>
<span class='chip' style='color:{COLORS["Asteroid"]}'>☄️ Asteroid rotation</span>
<span class='chip' style='color:{COLORS["Other"]}'>⭕ Other stars</span></div></div>"""

theme = gr.themes.Base(primary_hue="violet", neutral_hue="slate").set(
    body_background_fill=BG, block_background_fill=PANEL, block_border_color="#2a3352", body_text_color=INK,
    input_background_fill="#0b1020", button_primary_background_fill="linear-gradient(90deg,#7c3aed,#2dd4bf)",
    button_primary_text_color="white", block_label_text_color=MUTED, block_title_text_color=INK,
    checkbox_label_background_fill="#0b1020", checkbox_label_background_fill_selected="#2e1065",
    checkbox_label_text_color=INK, checkbox_label_text_color_selected="#ffffff", checkbox_label_border_color="#2a3352",
    background_fill_secondary=PANEL, border_color_primary="#2a3352")
_kw = dict(css=CSS, theme=theme)
blocks_kw = _kw if "css" in inspect.signature(gr.Blocks.__init__).parameters else {}
launch_kw = {} if blocks_kw else _kw

with gr.Blocks(title="Cosmic Detector", **blocks_kw) as demo:
    gr.HTML(HERO)
    with gr.Tab("🔭 Analyse"):
        with gr.Row():
            with gr.Column(scale=4):
                with gr.Tab("Real test data"):
                    cls = gr.Radio(SHORT, value="Lensing", label="Class")
                    pick = gr.Dropdown(gallery_choices("Lensing"), label="Light curve (never seen in training)")
                    go1 = gr.Button("Analyse", variant="primary")
                with gr.Tab("Upload your own"):
                    up = gr.File(label="Light-curve file (.dat, .txt, .csv)")
                    kind = gr.Radio(["OGLE phot.dat", "time, magnitude", "time, flux"], value="OGLE phot.dat", label="File format")
                    go2 = gr.Button("Analyse", variant="primary")
                phys = gr.HTML()
            with gr.Column(scale=6):
                verdict = gr.HTML("<div class='card'><div class='eyebrow'>VERDICT</div><div class='conf'>Pick a light curve and press Analyse.</div></div>")
                plot = gr.Plot(show_label=False)
        cls.change(lambda c: gr.update(choices=gallery_choices(c), value=None), cls, pick)
        go1.click(run_gallery, pick, [verdict, plot, phys])
        go2.click(run_upload, [up, kind], [verdict, plot, phys])
    with gr.Tab("📊 Model report"):
        gr.HTML(f"<div class='card'><div class='eyebrow'>TEST-SET RESULTS (objects never seen in training)</div>"
                f"<table class='rep'><tr><th>Model</th><th>Macro-F1</th><th>Balanced accuracy</th></tr>{report_rows}</table></div>"
                f"<div class='card'><div class='eyebrow'>HARD TESTS INSIDE ONE TELESCOPE</div>"
                f"<table class='rep'><tr><th>Task</th><th>Found</th><th>False alarms</th><th>PR-AUC</th></tr>{within_rows}</table>"
                f"<div class='note'>Our tE fits agree with OGLE's published values within 20% for {100*R.get('tE_within_20pct',0):.0f}% of test events. "
                f"{R.get('bh_candidates','?')} OGLE events have tE > 100 days (black-hole candidates).</div></div>")
        for f, cap in [("confusion_matrix.png", "Confusion matrix and PR-AUC"), ("lens_physics.png", "Lensing physics check")]:
            if os.path.exists(os.path.join(SAVE_DIR, f)):
                gr.Image(os.path.join(SAVE_DIR, f), label=cap)
    with gr.Tab("🧠 How it works"):
        gr.HTML("""<div class='card'><div class='eyebrow'>PIPELINE</div>
<div class='kv'><span>1 · Real data</span><b>Kepler (Kaggle) + OGLE-IV + ALCDEF (NASA PDS)</b></div>
<div class='kv'><span>2 · Same cleaning for all</span><b>brightness → spike removal → 1,024 bins</b></div>
<div class='kv'><span>3 · Two views</span><b>whole curve + high-pass (fast dips)</b></div>
<div class='kv'><span>4 · 1D ResNet CNN</span><b>shapes → 4 class probabilities</b></div>
<div class='kv'><span>5 · Physics stage</span><b>Paczyński tE · Lomb-Scargle · BLS</b></div>
<div class='note'>Splits are by object (a star or asteroid never appears in both train and test). Known limits: only 42 real planets exist in the Kepler set;
each class comes mainly from one survey, so we report results inside each telescope separately.</div></div>""")

demo.launch(share=IN_COLAB, **launch_kw)